# pyaegean neural lemmatizer — GreTa seq2seq (SOTA route)

Fine-tune **`bowphs/GreTa`** (pretrained Ancient-Greek char-level T5, the SOTA lemmatizer) to **generate** the lemma from the form. Generation handles unseen forms where our edit-tree classifier capped at 58.2%; literature puts GreTa ~91% F1. Target: unseen > stanza's 62.8%.

**Loop:** GPU runtime → Run all → **cell 6b prints `DEV lemma all/UNSEEN`** → (optionally) download `spike_model.zip`. torch/transformers used only here; inference is onnxruntime.

In [ ]:
!nvidia-smi -L  # MUST list a GPU
%pip -q install 'transformers>=4.46' 'datasets>=2.19' 'optimum[onnxruntime]>=1.20' accelerate sentencepiece protobuf onnx onnxruntime

## 0 · GPU + precision

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU, reconnect.'
USE_BF16 = torch.cuda.is_bf16_supported()
BS = 64 if USE_BF16 else 16
print(f'torch {torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)} | bf16={USE_BF16}')

## 1 · Upload `spike_data.zip`

In [ ]:
import json, zipfile, pathlib
from google.colab import files
up = files.upload()  # pick spike_data.zip
zipfile.ZipFile(next(n for n in up if n.endswith('.zip'))).extractall('.')
DATA = pathlib.Path('data') if pathlib.Path('data/train.jsonl').exists() else pathlib.Path('.')
train_rows = [json.loads(l) for l in open(DATA / 'train.jsonl', encoding='utf-8')]
print('form->lemma pairs:', len(train_rows), '| e.g.', train_rows[0])

## 2 · Tokenizer + model (`bowphs/GreTa`, T5 encoder-decoder)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
MODEL = 'bowphs/GreTa'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

## 3 · Tokenize (form -> input_ids, lemma -> labels)

In [ ]:
from datasets import Dataset
ML = 32
def prep(b):
    enc = tokenizer(b['form'], max_length=ML, truncation=True)
    enc['labels'] = tokenizer(text_target=b['lemma'], max_length=ML, truncation=True)['input_ids']
    return enc
ds = Dataset.from_list(train_rows).map(prep, batched=True, remove_columns=['form', 'lemma'])
ds = ds.train_test_split(test_size=0.02, seed=0)
print(ds)

## 4 · Fine-tune (Seq2SeqTrainer; best epoch by exact-match)
T5 uses **bf16, not fp16** (fp16 overflows); falls back to fp32 on a T4. ~10 epochs, a few minutes on an H100. Watch `exact` (exact-lemma match on the held-out slice) climb.

In [ ]:
import numpy as np
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)
collator = DataCollatorForSeq2Seq(tokenizer, model=model)
def compute_metrics(ep):
    preds, labels = ep
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    dp = tokenizer.batch_decode(preds, skip_special_tokens=True)
    dl = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {'exact': float(np.mean([a.strip() == b.strip() for a, b in zip(dp, dl)]))}
args = Seq2SeqTrainingArguments(
    output_dir='out', learning_rate=3e-4, lr_scheduler_type='cosine',
    per_device_train_batch_size=BS, per_device_eval_batch_size=BS*2,
    num_train_epochs=10, weight_decay=0.01, warmup_ratio=0.06,
    bf16=USE_BF16, fp16=False, tf32=USE_BF16, optim='adamw_torch_fused',
    dataloader_num_workers=2, predict_with_generate=True, generation_max_length=ML,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='exact', greater_is_better=True,
    logging_steps=100, report_to='none')
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=ds['train'],
                         eval_dataset=ds['test'], data_collator=collator,
                         processing_class=tokenizer, compute_metrics=compute_metrics)
trainer.train()
trainer.save_model('out_model'); tokenizer.save_pretrained('out_model')

## 6b · Dev lemma accuracy — GENERATE the lemma per form (the number that matters)

In [ ]:
import re, unicodedata
model.eval()
dev = [json.loads(l) for l in open(DATA / 'dev.jsonl', encoding='utf-8')]
forms = sorted({d['form'] for d in dev if d['scored']})
pred = {}
B = 256
for i in range(0, len(forms), B):
    b = forms[i:i+B]
    enc = tokenizer(b, return_tensors='pt', padding=True, truncation=True, max_length=ML).to(model.device)
    with torch.no_grad():
        g = model.generate(**enc, max_length=ML, num_beams=1)
    for f, d in zip(b, tokenizer.batch_decode(g, skip_special_tokens=True)):
        pred[f] = d
def _clean(s): return re.sub(r'\d+$', '', unicodedata.normalize('NFC', s))
na = nu = oa = ou = 0
for d in dev:
    if not d['scored']: continue
    na += 1; un = not d['seen']; nu += un
    if _clean(pred[d['form']]) == d['lemma']: oa += 1; ou += un
print(f'DEV lemma — all {oa/na:.1%}  UNSEEN {ou/nu:.1%}   '
      f'(beat: stanza 62.8% unseen; edit-tree 58.2%; pure-Python 40.3%)')

## 7 · Export ONNX (seq2seq) + download
`optimum` exports encoder + decoder for torch-free `ORTModelForSeq2SeqLM.generate()`. The zip is large (fp32 T5) — the **cell-6b number above is the answer**; the zip is only needed for the local onnxruntime confirmation.

In [ ]:
import os, shutil
!optimum-cli export onnx --model out_model --task text2text-generation onnx 2>/dev/null || \
 optimum-cli export onnx --model out_model --task text2text-generation onnx
sz = sum(os.path.getsize(os.path.join('onnx', f)) for f in os.listdir('onnx')) // (1024*1024)
print('onnx dir', sz, 'MB:', sorted(os.listdir('onnx')))
shutil.make_archive('spike_model', 'zip', 'onnx')
files.download('spike_model.zip')

## Next
Report cell 6b's `DEV lemma all/UNSEEN`. **Unseen > 62.8% = we beat stanza** on the clean metric (we already beat it on overall). Send `spike_model.zip` for the local onnxruntime confirmation: `python eval_seq2seq.py --model <unzipped> --dev data/dev.jsonl`.